In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import yaml

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.oasisb.oasisb_utils import (
    EM_EDAX_MIME_TYPES_SIDECAR,  # edax
    EM_EDAX_MIME_TYPES_SOLITARY,
    # APM_MIME_TYPES_SIDECAR,  # only for cross-referencing between apm and em collections
    # APM_MIME_TYPES_SOLITARY,
    EM_HFIVE_MIME_TYPES_SIDECAR,  # hdf
    EM_HFIVE_MIME_TYPES_SOLITARY,
    EM_IMAGE_MIME_TYPES_SIDECAR,  # image
    EM_IMAGE_MIME_TYPES_SOLITARY,
    EM_KPY_MIME_TYPES_SIDECAR,  # kikuchipy diffraction pattern
    EM_KPY_MIME_TYPES_SOLITARY,
    EM_MIXED_MIME_TYPES_SIDECAR,  # mixed, spectrum, etc.
    EM_MIXED_MIME_TYPES_SOLITARY,
    EM_MTEX_MIME_TYPES_SIDECAR,  # mtex
    EM_MTEX_MIME_TYPES_SOLITARY,
    get_project_id,
    prepare_parsing,
    prepare_parsing_via_config_file,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

## Decompress the original files from the scientists from the storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 880)

with open(f"{src_directory}{os.sep}aaa_em_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

# keep_searching_toggle = True
count: int = 0  # how many files to decompress
volume: int = 0  # how much byte volume does this add to scratch
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                if project_id in nomad_project_names:
                    status = prepare_parsing(
                        f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                        src_directory,
                        project_id,
                        trg_directory,
                        report=True,
                        write=False,
                        mime_type="image",  # "image",  # "mtex", "hdf", "image", "mixed"
                        mime_type_solitary=EM_IMAGE_MIME_TYPES_SOLITARY,
                        mime_type_sidecar=EM_IMAGE_MIME_TYPES_SIDECAR,
                    )
                    for key, obj in status.items():
                        if obj["n"] > 0:
                            print(f"{project_id}, {key}, {obj['n']}, {obj['bytes']}")
                            count += obj["n"]
                            volume += obj["bytes"]
print(f"Batch queue completed, {count} files, {volume / 1024**3} GiB uncompressed")

***

Programmatic identification of atom_types from file names the collection has more than 300k files.<br>
So we use sampling at most 10 

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
from odf.style import Style, TableColumnProperties
from odf.table import Table, TableCell, TableColumn, TableRow
from odf.text import P


def make_text_cell(text):
    cell = TableCell(valuetype="string")
    cell.setAttribute("stringvalue", text)
    cell.addElement(P(text=text))
    return cell


def generate_human_annotable_spreadsheet(
    file_path_prefix: str,
    project_id: str,
    candidates: list[tuple[str, str]],
    mime_type: str = "",
) -> None:
    doc = OpenDocumentSpreadsheet()

    table = Table(name=f"{project_id}")

    row = TableRow()
    for col_name, col_width in [
        # ("atom_types", 10),
        ("src", 16),
        ("trg", 16),
    ]:
        col_style = Style(name=col_name, family="table-column")
        col_style.addElement(TableColumnProperties(columnwidth=f"{col_width}in"))
        doc.automaticstyles.addElement(col_style)
        table.addElement(TableColumn(stylename=col_style))

        # cell = TableCell()
        # cell.addElement()  # P(text=col_name))
        row.addElement(make_text_cell(col_name))
    table.addElement(row)

    for src, trg in candidates:
        row = TableRow()

        # cell = TableCell()
        # cell.addElement(P(text=""))
        # row.addElement(cell)
        # cell = TableCell()
        # cell.addElement(make_text_cell(src))  # P(text=src))
        row.addElement(make_text_cell(src))  # cell)
        # cell = TableCell()
        # cell.addElement(make_text_cell(trg))  # P(text=trg))
        row.addElement(make_text_cell(trg))  # cell)

        table.addElement(row)

    doc.spreadsheet.addElement(table)
    doc.save(
        f"{file_path_prefix}{os.sep}{project_id}{f'''.{mime_type}''' if mime_type != '' else ''}.decompressed.csv.subset.ods"
    )

In [ ]:
# pattern = os.path.join(f"{trg_directory}{os.sep}049.mixed.decompressed.csv")
pattern = os.path.join(f"{trg_directory}{os.sep}*.image.decompressed.csv")
decompression_logfiles: list[str] = glob.glob(pattern)
statistics: dict[str, int] = {}
maximum: int = 10
total: int = 0
for file in sorted(decompression_logfiles):
    project_id = file.rsplit(os.sep, 1)[1].split(".", 1)[0]
    print(project_id)
    with open(file, encoding="utf-8") as fp:
        # statistics[file] = sum(1 for _ in fp) - 3  # three header lines
        # fp.seek(0)

        # hashing required because of sidecar files appearing also in the list
        # by convention sidecar files have the same filename stem as the main file
        # e.g. ("jeol.tif", "jeol.txt"), ("jeol.bmp", "jeol.txt") but there are also trickier cases
        # like these ("tescan.tif", "tescan-tif.hdr")
        candidate_lookup: dict[str, list[tuple[str, str]]] = {}
        hashes: set[str] = set()
        for line in fp:
            parts = line.strip().split(";")
            if len(parts) == 5:
                hsh = parts[4].rsplit(os.sep, 1)[1][4:].split(".")[0]
                hashes.add(hsh)
                if hsh not in candidate_lookup:
                    candidate_lookup[hsh] = [(parts[2], parts[4])]
                else:
                    candidate_lookup[hsh].append((parts[2], parts[4]))

        rng = np.random.default_rng(seed=int(project_id))  # deterministic seed
        candidate_hashes = list(hashes)
        sample_n: int = (
            maximum if len(candidate_hashes) > maximum else len(candidate_hashes)
        )
        print(f"sample_n {sample_n}")
        selected_hashes = rng.choice(list(hashes), size=sample_n, replace=False)

        # collect all files with the selected hashes, given the above-mentioned assumptions
        # that will include again sidecar files if present
        selected: list[tuple[str, str]] = []
        for hsh in selected_hashes:
            for src, trg in candidate_lookup[hsh]:
                selected.append((src, trg))
                # print(f">>>>{src}")
                # print(f"<<<<{trg}")
        total += len(selected_hashes)

        generate_human_annotable_spreadsheet(
            trg_directory, project_id, selected, mime_type="image"
        )

        del (
            candidate_lookup,
            hashes,
            parts,
            hsh,
            rng,
            candidate_hashes,
            sample_n,
            selected_hashes,
            selected,
        )

        prepare_parsing_via_config_file(
            f"{trg_directory}{os.sep}{project_id}.image.decompressed.csv.subset.ods",
            src_directory,
            project_id,
            trg_directory,
            report=True,
            write=True,
            mime_type="image",  # "image",  # "mtex", "hdf", "image", "mixed"
        )

print(f"total {total}")

In [ ]:
project_id = "009"

***

In [ ]:
config_file = pd.read_excel(
    f"{os.getcwd()}{os.sep}009.config.image.ods",
    sheet_name="009",
    engine="odf",
    dtype=str,
).fillna("")
print(config_file)

In [ ]:
"""
count: int = 0
for key, value in sorted(statistics.items(), key=lambda item: item[1], reverse=True):
    print(f"{value};{key.replace(f'''{trg_directory}{os.sep}''', '')}")
    if value > maximum:
        total += maximum
    else:
        total += value
    count += 1

"""